In [1]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

In [2]:
USER = "cs_user"
PASSWORD = "cs_password"     
HOST = "localhost"
DATABASE = "product_reviews"

In [3]:
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:3306/{DATABASE}")
print("Connected to MySQL successfully!")

Connected to MySQL successfully!


In [4]:
df = pd.read_csv("../data/processed/amazon_reviews_cleaned.csv")
print("CSV Loaded! Rows:", len(df))

CSV Loaded! Rows: 67807


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67807 entries, 0 to 67806
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  67807 non-null  object             
 1   product_name        61112 non-null  object             
 2   asins               67807 non-null  object             
 3   brand               67807 non-null  object             
 4   category            67807 non-null  object             
 5   keywords            67807 non-null  object             
 6   manufacturer        67807 non-null  object             
 7   review_date         67807 non-null  datetime64[ns, UTC]
 8   reviews_dateSeen    67807 non-null  object             
 9   do_recommend        55152 non-null  float64            
 10  reviews_numHelpful  55214 non-null  float64            
 11  rating              67807 non-null  int64              
 12  reviews_sourceURLs  67807 non-nu

In [6]:
# Ensure date formatting
if "review_date" in df.columns:
    df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")

In [8]:
df = df.rename(columns={
    "reviews.dateSeen": "reviews_dateSeen",
    "reviews.numHelpful": "reviews_numHelpful",
    "reviews.sourceURLs": "reviews_sourceURLs",
    "reviews.username": "reviews_username"
})

In [10]:
df = df.rename(columns={"keys":"keywords"})

In [13]:
df['reviews_dateSeen'] = df['reviews_dateSeen'].astype(str)

# Take the FIRST date if multiple exist
df['reviews_dateSeen'] = df['reviews_dateSeen'].apply(
    lambda x: x.split(',')[0] if ',' in x else x
)

# Convert to datetime (force errors to NaT)
df['reviews_dateSeen'] = pd.to_datetime(
    df['reviews_dateSeen'],
    errors='coerce',
    utc=True
)

# remove timezone for MySQL compatibility
df['reviews_dateSeen'] = df['reviews_dateSeen'].dt.tz_localize(None)

df[['reviews_dateSeen']].head()

,reviews_dateSeen
0,2017-06-07 09:04:00
1,2017-06-07 09:04:00
2,2017-06-07 09:04:00
3,2017-06-07 09:04:00
4,2017-06-07 09:04:00


In [17]:
print("Inserting data into MySQL...")

df.to_sql(
    name="amazon_reviews",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Data inserted successfully!")

Inserting data into MySQL...
Data inserted successfully!
